## Basic Definitions in Simple Words

| Term | Meaning in simple words |
|---|---|
| **Text preprocessing** | Every step that turns raw text into the tokens a retriever stores and compares. |
| **Token** | One unit of text after splitting, usually a word, number or code like `e401`. |
| **Tokenization** | Cutting text into tokens. `"reset password"` becomes `["reset", "password"]`. |
| **Normalization** | Making different spellings of the same thing identical, for example `Café`, `CAFE` and `cafe` all become `cafe`. |
| **Unicode normalization** | Making characters that look the same also have the same bytes. `ﬁ` (one ligature character) becomes `fi`. |
| **Case folding** | Turning text to one case so `Password` and `password` match. |
| **Accent folding** | Removing accents so `résumé` matches `resume`. |
| **Stop words** | Very common words such as `the`, `is`, `of` that carry little meaning on their own. |
| **Stemming** | Chopping word endings by rules: `running`, `runs` become `run`. Fast, sometimes produces non-words. |
| **Lemmatization** | Mapping a word to its dictionary form using a vocabulary: `ran` becomes `run`, `better` becomes `good`. |
| **N-gram** | A sequence of N neighbouring tokens (`new york`) or characters (`pas`, `ass`, `ssw`). |
| **Analyzer** | Search-engine name (Elasticsearch, OpenSearch, Lucene) for one complete preprocessing pipeline. |
| **Vocabulary mismatch** | Query and document mean the same thing but use different tokens, so lexical search misses it. |
| **Over-processing** | Cleaning so hard that different meanings collapse into the same token, for example removing `not`. |

## 1. Where Preprocessing Fits in RAG

Preprocessing runs twice: once on every document at index time and once on every query at search time. Retrieval can only match what these two runs produce.

```mermaid
%%{init: {'theme':'base','themeVariables':{'primaryColor':'#EEF2FF','primaryBorderColor':'#6366F1','primaryTextColor':'#1E1B4B','lineColor':'#6366F1'}}}%%
flowchart LR
    subgraph IDX["Index time"]
        direction TB
        D["Raw documents"] --> P1["Preprocess"] --> T1["Document<br/>tokens"] --> I[("Index")]
    end
    subgraph QRY["Query time"]
        direction TB
        Q["User query"] --> P2["Same<br/>preprocess"] --> T2["Query<br/>tokens"]
    end
    T2 --> M{"Match<br/>tokens"}
    I --> M
    M --> K["Top-K<br/>chunks"]
    style IDX fill:#EFF6FF,stroke:#2563EB
    style QRY fill:#F0FDF4,stroke:#16A34A
    style M fill:#FEF3C7,stroke:#D97706
```

> [!TIP]
> **In simple words:** a lexical retriever compares strings. If the document says `Password` and the user types `password`, the strings differ unless preprocessing made them equal.

Where it matters most:

| Retriever | How much preprocessing matters | Why |
|---|---|---|
| **BM25 / TF-IDF (sparse)** | A lot | It matches exact tokens, so every spelling difference is a miss. |
| **Dense embeddings** | Less, but not zero | The embedding model has its own tokenizer; you mainly clean noise and keep text natural. |
| **Hybrid search** | Both | The sparse branch needs the analyzer; the dense branch needs clean, natural text. |

## 2. Why Preprocessing Matters for Retrieval

Users never type text exactly the way documents are written. Without preprocessing, exact matching fails on trivial differences.

In [1]:
import re
import unicodedata
from collections import Counter

# Document text and four ways a user might ask about it
document = "Reset your Password after error E-401."
queries = ["password", "PASSWORD", "e401", "error e-401"]

# Naive tokens: split on spaces only, keep case and punctuation
doc_tokens = set(document.split())

for query in queries:
    # Which query words appear exactly in the document?
    hits = [word for word in query.split() if word in doc_tokens]

    print(f"{query!r:<15} matches: {hits}")

'password'      matches: []
'PASSWORD'      matches: []
'e401'          matches: []
'error e-401'   matches: ['error']


Only `error` matched. `Password`, `PASSWORD`, `E-401.` and `e401` are four different strings to a computer. The rest of this chapter fixes each cause one step at a time.

```mermaid
%%{init: {'theme':'base','themeVariables':{'primaryColor':'#EEF2FF','primaryBorderColor':'#6366F1','primaryTextColor':'#1E1B4B','lineColor':'#6366F1'}}}%%
flowchart LR
    A["Case<br/>Password / password"] --> MISS(["Lexical<br/>miss"])
    B["Punctuation<br/>E-401. / e401"] --> MISS
    C["Accents<br/>résumé / resume"] --> MISS
    D["Word forms<br/>running / run"] --> MISS
    E["Unicode look-alikes<br/>ﬁle / file"] --> MISS
    classDef c fill:#FEE2E2,stroke:#DC2626,color:#7F1D1D,stroke-width:2px
    class A,B,C,D,E c
    style MISS fill:#F1F5F9,stroke:#475569
```

## 3. The Golden Rule: Same Pipeline on Both Sides

> [!IMPORTANT]
> Documents and queries must go through the **same** preprocessing function. A mismatch silently returns nothing, and no error is raised.

```mermaid
%%{init: {'theme':'base','themeVariables':{'primaryColor':'#EEF2FF','primaryBorderColor':'#6366F1','primaryTextColor':'#1E1B4B','lineColor':'#6366F1'}}}%%
sequenceDiagram
    autonumber
    participant D as Document
    participant A as Analyzer
    participant I as Index
    participant U as User
    D->>A: "Café Menu"
    A->>I: [cafe, menu]
    U->>A: "cafe menu"
    A-->>U: [cafe, menu]
    U->>I: lookup cafe, menu
    I-->>U: match found
    Note over A: One analyzer object,<br/>used at index and query time
```

The bug in code: the index lowercases and folds accents, the query side forgot to.

In [2]:
def analyzer(text):
    # Lowercase, fold accents, keep letters and digits
    text = unicodedata.normalize("NFKD", text.lower())
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    return re.findall(r"[a-z0-9]+", text)


# Index time: documents go through the analyzer
index_tokens = set(analyzer("Café Menu and Prices"))

# Query time, correct: same analyzer
good = [t for t in analyzer("café prices") if t in index_tokens]

# Query time, buggy: raw split, no analyzer
bad = [t for t in "café prices".split() if t in index_tokens]

print("index tokens :", sorted(index_tokens))
print("same analyzer:", good)
print("no analyzer  :", bad)

index tokens : ['and', 'cafe', 'menu', 'prices']
same analyzer: ['cafe', 'prices']
no analyzer  : ['prices']


## 4. Unicode Normalization

The same visible character can be stored in different ways. PDFs and web pages are full of them: ligatures, full-width letters, non-breaking spaces and accents stored as two code points.

| Form | What it does | Use for retrieval? |
|---|---|---|
| **NFC** | Combines letter + accent into one code point | Safe default for display text |
| **NFD** | Splits a letter and its accent into two code points | Step before accent folding |
| **NFKC** | NFC plus "compatibility" mapping: `ﬁ` to `fi`, `Ｅ` to `E`, no-break space to space | Best for the search index |
| **NFKD** | NFD plus compatibility mapping | Before accent folding on the index side |

In [3]:
samples = {
    "ligature": "ﬁle",
    "full-width": "Ｅ４０１",
    "decomposed accent": "café",
    "no-break space": "price: 10",
}

for name, raw in samples.items():
    # Compatibility normalization turns look-alikes into plain characters
    clean = unicodedata.normalize("NFKC", raw)

    print(f"{name:<18} {raw!r:<14} -> {clean!r:<10} length {len(raw)} -> {len(clean)}")

# Composed and decomposed "café" look identical but are different strings
print("\ncomposed == decomposed before:", "café" == "café")
print("composed == decomposed after :", unicodedata.normalize("NFC", "café") == unicodedata.normalize("NFC", "café"))

ligature           'ﬁle'          -> 'file'     length 3 -> 4
full-width         'Ｅ４０１'         -> 'E401'     length 4 -> 4
decomposed accent  'café'        -> 'café'     length 5 -> 4
no-break space     'price:\xa010' -> 'price: 10' length 9 -> 9

composed == decomposed before: False
composed == decomposed after : True


> [!TIP]
> **In simple words:** normalize Unicode first, before any other step. Every later step (lowercasing, regex tokenizing) behaves predictably only on normalized text.

## 5. Case Folding

`lower()` handles English. `casefold()` is the stronger, language-aware version used for matching.

In [4]:
for word in ["Password", "PASSWORD", "Straße", "ΣΊΣΥΦΟΣ"]:
    # lower() for display, casefold() for matching
    print(f"{word:<10} lower={word.lower():<10} casefold={word.casefold()}")

# German: casefold makes "Straße" and "STRASSE" match
print("\nStraße vs STRASSE matched with casefold:", "Straße".casefold() == "STRASSE".casefold())

Password   lower=password   casefold=password
PASSWORD   lower=password   casefold=password
Straße     lower=straße     casefold=strasse
ΣΊΣΥΦΟΣ    lower=σίσυφος    casefold=σίσυφοσ

Straße vs STRASSE matched with casefold: True


> [!WARNING]
> Case can carry meaning: `US` (country) vs `us`, `IT` (department) vs `it`, `Apple` vs `apple`. Most retrieval systems still fold case, because the misses it fixes are far more common than these collisions. A dense retriever or a reranker can recover the meaning later.

## 6. Accent Folding

Users rarely type accents. Folding them on both sides makes `résumé`, `resumé` and `resume` one token.

In [5]:
def fold_accents(text):
    # Split letters from their accent marks
    decomposed = unicodedata.normalize("NFKD", text)

    # Drop the accent marks (combining characters)
    return "".join(ch for ch in decomposed if not unicodedata.combining(ch))


for word in ["résumé", "Café", "naïve", "São Paulo", "Ångström"]:
    print(f"{word:<10} -> {fold_accents(word)}")

résumé     -> resume
Café       -> Cafe
naïve      -> naive
São Paulo  -> Sao Paulo
Ångström   -> Angstrom


> [!NOTE]
> In some languages an accent changes the word (Spanish `año` "year" vs `ano`). For multilingual search, index both the folded and the original form, which Elasticsearch does with its `asciifolding` filter and `preserve_original`.

## 7. Whitespace, Line Breaks and Hyphenation

Extracted PDF text often contains broken lines, hyphenated words split across lines and repeated spaces.

In [6]:
pdf_text = "The informa-\ntion retrieval   system\nhandles  multi-\nline text.\t\tDone."

# Join words hyphenated at a line break: "informa-\ntion" -> "information"
text = re.sub(r"(\w)-\n(\w)", r"\1\2", pdf_text)

# Every run of whitespace becomes one space
text = re.sub(r"\s+", " ", text).strip()

print(repr(pdf_text))
print(repr(text))

'The informa-\ntion retrieval   system\nhandles  multi-\nline text.\t\tDone.'
'The information retrieval system handles multiline text. Done.'


Note that `multi-\nline` became `multiline`. A real hyphen at the end of a line cannot be told apart from a split word without a dictionary; most pipelines accept this small error.

## 8. Tokenization for Retrieval

The tokenizer decides what a "word" is. For RAG the hardest cases are codes, versions, part numbers and symbols, which are often exactly what users search for.

In [7]:
text = "Install Python 3.12 for part 6ES7513-1AL02-0AB0; read the C++ guide or mail support@example.com."

# 1. Split on spaces: punctuation stays glued to words
split_tokens = text.lower().split()

# 2. Letters and digits only: codes break into pieces
simple_tokens = re.findall(r"[a-z0-9]+", text.lower())

# 3. Code-aware: keep . - @ inside a token, keep a trailing ++
CODE_TOKEN = r"[a-z0-9]+(?:[.\-@][a-z0-9]+)*(?:\+\+)?"
code_tokens = re.findall(CODE_TOKEN, text.lower())

print("split      :", split_tokens)
print("simple     :", simple_tokens)
print("code-aware :", code_tokens)

split      : ['install', 'python', '3.12', 'for', 'part', '6es7513-1al02-0ab0;', 'read', 'the', 'c++', 'guide', 'or', 'mail', 'support@example.com.']
simple     : ['install', 'python', '3', '12', 'for', 'part', '6es7513', '1al02', '0ab0', 'read', 'the', 'c', 'guide', 'or', 'mail', 'support', 'example', 'com']
code-aware : ['install', 'python', '3.12', 'for', 'part', '6es7513-1al02-0ab0', 'read', 'the', 'c++', 'guide', 'or', 'mail', 'support@example.com']


| Tokenizer | `3.12` | `6ES7513-1AL02-0AB0` | `C++` | Risk |
|---|---|---|---|---|
| **Split on spaces** | `3.12` | kept | `c++` | `0ab0;` and `support@example.com.` keep punctuation, so they never match |
| **Letters and digits** | `3`, `12` | `6es7513`, `1al02`, `0ab0` | `c` | Versions and codes lose identity; `3.12` also matches `3.11 ... 12` |
| **Code-aware** | `3.12` | `6es7513-1al02-0ab0` | `c++` | A query typed without the hyphens no longer matches |

> [!TIP]
> Common production answer: index **both** forms. Keep the code-aware token for exact matches and also the split pieces for partial matches. Search engines call this a multi-field or `word_delimiter_graph` setup.

## 9. Stop Words

Stop words are frequent words such as `the`, `a`, `of`, `is`. Removing them shrinks the sparse index and speeds up search. BM25 already gives them a very low IDF, so removal changes ranking less than people expect.

In [8]:
STOP_WORDS = {"the", "a", "an", "of", "to", "is", "are", "and", "or", "in", "on", "for", "not", "be", "this", "that", "with", "under"}

paragraph = ("The regulation of the European Union is a set of rules for the safety of products "
             "that are sold in the market and the duties of the suppliers of those products")

# Tokens before and after stop-word removal
tokens = re.findall(r"[a-z]+", paragraph.lower())
kept = [t for t in tokens if t not in STOP_WORDS]

print("tokens before:", len(tokens))
print("tokens after :", len(kept))
print("kept:", kept)

tokens before: 31
tokens after : 13
kept: ['regulation', 'european', 'union', 'set', 'rules', 'safety', 'products', 'sold', 'market', 'duties', 'suppliers', 'those', 'products']


```mermaid
%%{init: {'theme':'base','themeVariables':{'primaryColor':'#EEF2FF','primaryBorderColor':'#6366F1','primaryTextColor':'#1E1B4B','lineColor':'#6366F1'}}}%%
xychart-beta
    title "Tokens in one sentence"
    x-axis ["All tokens", "Without stop words"]
    y-axis "tokens" 0 --> 35
    bar [31, 13]
```

The danger: some stop words carry meaning.

In [9]:
a = "Products are prohibited under Article 33"
b = "Products are not prohibited under Article 33"

# Same stop-word removal on both sentences
clean_a = [t for t in re.findall(r"[a-z0-9]+", a.lower()) if t not in STOP_WORDS]
clean_b = [t for t in re.findall(r"[a-z0-9]+", b.lower()) if t not in STOP_WORDS]

print(clean_a)
print(clean_b)
print("identical after removal:", clean_a == clean_b)

['products', 'prohibited', 'article', '33']
['products', 'prohibited', 'article', '33']
identical after removal: True


> [!WARNING]
> Never remove negations (`not`, `no`, `without`) in legal, medical or compliance RAG. Phrase queries like `to be or not to be` disappear completely with a standard stop list. Many modern systems keep all words in the index and rely on IDF instead.

## 10. Stemming vs Lemmatization

Both reduce word forms so `running`, `runs` and `run` can match.

```mermaid
%%{init: {'theme':'base','themeVariables':{'primaryColor':'#EEF2FF','primaryBorderColor':'#6366F1','primaryTextColor':'#1E1B4B','lineColor':'#6366F1'}}}%%
flowchart LR
    W["running, runs, ran"] --> S["Stemmer<br/>rule-based suffix cut"]
    W --> L["Lemmatizer<br/>dictionary lookup"]
    S --> S1["run, run, ran"]
    L --> L1["run, run, run"]
    style S fill:#DBEAFE,stroke:#2563EB
    style L fill:#DCFCE7,stroke:#16A34A
```

In [10]:
from nltk.stem import PorterStemmer, SnowballStemmer, WordNetLemmatizer

porter = PorterStemmer()
snowball = SnowballStemmer("english")
lemmatizer = WordNetLemmatizer()

words = ["running", "runs", "ran", "studies", "studying", "better", "organization", "organ", "university", "universe"]

print(f"{'word':<14}{'porter':<12}{'snowball':<12}{'lemma (verb)':<14}{'lemma (adj)'}")

for w in words:
    # Rule-based stems and dictionary lemmas side by side
    print(f"{w:<14}{porter.stem(w):<12}{snowball.stem(w):<12}{lemmatizer.lemmatize(w, 'v'):<14}{lemmatizer.lemmatize(w, 'a')}")

word          porter      snowball    lemma (verb)  lemma (adj)
running       run         run         run           running
runs          run         run         run           runs
ran           ran         ran         run           ran
studies       studi       studi       study         studies
studying      studi       studi       study         studying
better        better      better      better        good
organization  organ       organ       organization  organization
organ         organ       organ       organ         organ
university    univers     univers     university    university
universe      univers     univers     universe      universe


Read the output carefully:

| Observation | Example from the output | Consequence |
|---|---|---|
| Stems are not always words | `studies` to `studi` | Fine for matching, ugly for display |
| Stemmers miss irregular forms | `ran` stays `ran` | Lemmatizer fixes it |
| Stemmers over-merge | `university` and `universe` both become `univers` | False matches |
| Lemmatizer needs the part of speech | `better` becomes `good` only as an adjective | Needs a POS tagger in practice |

> [!TIP]
> **In simple words:** for BM25 in English, a light stemmer (Snowball or Porter) is the usual choice: cheap and good enough. Use lemmatization when display quality or precision matters. Do **not** stem text that goes to an embedding model.

## 11. N-grams for Phrases and Typos

Word n-grams keep phrases together. Character n-grams make search tolerant to typos.

In [11]:
def word_ngrams(tokens, n):
    # Every run of n neighbouring tokens
    return [" ".join(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]


def char_ngrams(word, n=3):
    # Pad so the first and last letters get their own n-grams
    padded = f"#{word}#"
    return {padded[i:i + n] for i in range(len(padded) - n + 1)}


def jaccard(a, b):
    # Shared n-grams divided by all n-grams
    return len(a & b) / len(a | b)


print("word bigrams:", word_ngrams(["new", "york", "pizza", "places"], 2))

for typo in ["pasword", "passwrod", "banana"]:
    score = jaccard(char_ngrams("password"), char_ngrams(typo))
    print(f"password vs {typo:<9} char-trigram Jaccard = {score:.2f}")

word bigrams: ['new york', 'york pizza', 'pizza places']
password vs pasword   char-trigram Jaccard = 0.67
password vs passwrod  char-trigram Jaccard = 0.33
password vs banana    char-trigram Jaccard = 0.00


```math
J(A, B) = \frac{|A \cap B|}{|A \cup B|}
```

The typo `pasword` still shares two thirds of its trigrams with `password` (0.67), a swapped-letter typo shares a third (0.33), and an unrelated word shares none. Search engines use this idea for fuzzy matching and "did you mean" suggestions.

## 12. Protecting Special Tokens

Aggressive cleaning destroys exactly the tokens that make RAG queries precise: versions, IDs, amounts, emails. Protect them before the generic steps run.

In [12]:
PROTECTED = re.compile(r"[a-z0-9]+(?:[.\-@/][a-z0-9]+)+")


def preprocess_naive(text):
    # Lowercase and keep runs of letters only: digits and codes are lost
    return re.findall(r"[a-z]+", text.lower())


def preprocess_protected(text):
    text = text.lower()

    # Pull out codes, versions and emails first, unchanged
    keep = PROTECTED.findall(text)
    rest = PROTECTED.sub(" ", text)

    # Generic tokens from the remaining text
    return keep + re.findall(r"[a-z0-9]+", rest)


sample = "Upgrade to Python 3.12 before 2025-01-31; ticket INC-4521 by ops@example.com"

print("naive    :", preprocess_naive(sample))
print("protected:", preprocess_protected(sample))

naive    : ['upgrade', 'to', 'python', 'before', 'ticket', 'inc', 'by', 'ops', 'example', 'com']
protected: ['3.12', '2025-01-31', 'inc-4521', 'ops@example.com', 'upgrade', 'to', 'python', 'before', 'ticket', 'by']


## 13. A Configurable Preprocessing Pipeline

All steps in one class, each switchable, so documents and queries share one object.

```mermaid
%%{init: {'theme':'base','themeVariables':{'primaryColor':'#EEF2FF','primaryBorderColor':'#6366F1','primaryTextColor':'#1E1B4B','lineColor':'#6366F1'}}}%%
classDiagram
    class Preprocessor {
        +bool unicode_nfkc
        +bool casefold
        +bool fold_accents
        +bool protect_codes
        +bool remove_stop_words
        +str stemmer
        +__call__(text) list
    }
    class Index {
        +add(doc_tokens)
    }
    class Query {
        +tokens
    }
    Preprocessor <.. Index : same object
    Preprocessor <.. Query : same object
```

```mermaid
%%{init: {'theme':'base','themeVariables':{'primaryColor':'#EEF2FF','primaryBorderColor':'#6366F1','primaryTextColor':'#1E1B4B','lineColor':'#6366F1'}}}%%
stateDiagram-v2
    [*] --> Unicode : raw text
    Unicode --> Whitespace : NFKC
    Whitespace --> Case : clean spaces
    Case --> Accents : casefold
    Accents --> Tokens : fold accents
    Tokens --> StopWords : code-aware split
    StopWords --> Stems : optional removal
    Stems --> [*] : stem, keep codes
```

In [13]:
from dataclasses import dataclass


@dataclass
class Preprocessor:
    unicode_nfkc: bool = True
    casefold: bool = True
    fold_accents: bool = True
    protect_codes: bool = True
    remove_stop_words: bool = False
    stemmer: str = "snowball"

    def __call__(self, text):
        # 1. Same characters for look-alike characters
        if self.unicode_nfkc:
            text = unicodedata.normalize("NFKC", text)

        # 2. Join hyphenated line breaks, collapse whitespace
        text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)
        text = re.sub(r"\s+", " ", text)

        # 3. One case
        if self.casefold:
            text = text.casefold()

        # 4. Drop accents
        if self.fold_accents:
            text = fold_accents(text)

        # 5. Tokens: codes kept whole, or simple letters and digits
        pattern = CODE_TOKEN if self.protect_codes else r"[a-z0-9]+"
        tokens = re.findall(pattern, text)

        # 6. Optional stop-word removal
        if self.remove_stop_words:
            tokens = [t for t in tokens if t not in STOP_WORDS]

        # 7. Stem plain words only; leave codes and numbers untouched
        if self.stemmer == "snowball":
            tokens = [snowball.stem(t) if t.isalpha() else t for t in tokens]

        return tokens


analyze = Preprocessor()

print(analyze("Résumé writing TIPS for Running ﬁrst-time interviews"))
print(analyze("Install Python 3.12 (part 6ES7513-1AL02-0AB0)"))

['resum', 'write', 'tip', 'for', 'run', 'first-time', 'interview']
['instal', 'python', '3.12', 'part', '6es7513-1al02-0ab0']


## 14. Preprocessing for Sparse vs Dense Retrieval

| Step | BM25 / sparse | Dense embeddings | Why |
|---|---|---|---|
| **Unicode NFKC** | Yes | Yes | Removes invisible differences everywhere |
| **Whitespace / hyphenation fix** | Yes | Yes | Broken words hurt both |
| **Case folding** | Yes | Usually no | Most embedding models handle case themselves (MiniLM is uncased) |
| **Accent folding** | Yes | No | Models learned accented text; keep it natural |
| **Stop-word removal** | Optional | No | Embeddings need full sentences; "not" matters |
| **Stemming / lemmatization** | Often yes | No | Model tokenizers expect real words |
| **Protect codes and versions** | Yes | Yes | Both lose precision on IDs if they are mangled |

```mermaid
%%{init: {'theme':'base','themeVariables':{'primaryColor':'#EEF2FF','primaryBorderColor':'#6366F1','primaryTextColor':'#1E1B4B','lineColor':'#6366F1'}}}%%
quadrantChart
    title How much to preprocess
    x-axis Light cleaning --> Heavy normalization
    y-axis Natural text needed --> Exact tokens needed
    quadrant-1 BM25 analyzer
    quadrant-2 Keyword filters
    quadrant-3 Embedding input
    quadrant-4 Over-processing risk
    BM25 index: [0.8, 0.8]
    Dense embeddings: [0.2, 0.2]
    Reranker input: [0.15, 0.3]
    Metadata filters: [0.35, 0.85]
```

Real check with the dense model from group 06 (`all-MiniLM-L6-v2`, local run). Its tokenizer already lowercases and strips accents, so `Résumé` and `resume` give the identical vector (cosine 1.000). Word forms and versions are close but not identical, which is why the sparse side still needs stemming and why versions need exact tokens.

In [14]:
import os
os.environ["HF_HUB_OFFLINE"] = "1"

from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

pairs = [("Résumé writing tips", "resume writing tips"),
         ("PASSWORD RESET", "password reset"),
         ("running shoes", "run shoe"),
         ("Python 3.12", "Python 3.11")]

for a, b in pairs:
    # Cosine similarity of unit vectors = dot product
    va, vb = model.encode([a, b], normalize_embeddings=True)

    print(f"{a!r:<24} vs {b!r:<22} cosine = {float(va @ vb):.3f}")

'Résumé writing tips'    vs 'resume writing tips'  cosine = 1.000
'PASSWORD RESET'         vs 'password reset'       cosine = 1.000
'running shoes'          vs 'run shoe'             cosine = 0.903
'Python 3.12'            vs 'Python 3.11'          cosine = 0.899


## 15. End-to-End Example: Does Preprocessing Change BM25 Results?

Eight short documents, five queries written the way users really type (caps, no accents, base word forms). Each query has one correct document. Three pipelines feed the same BM25 index.

In [15]:
from rank_bm25 import BM25Okapi

documents = {
    "D1": "How to reset your Password after error E-401",
    "D2": "Résumé writing tips for software engineers",
    "D3": "Installing Python 3.12 on Windows",
    "D4": "Running shoes: a buyer's guide",
    "D5": "Products are not prohibited under Article 33",
    "D6": "Office opening hours and holidays",
    "D7": "Python 3.11 release notes",
    "D8": "Café menu and prices",
}

queries = {
    "PASSWORD e401": "D1",
    "resume": "D2",
    "install python 3.12": "D3",
    "run shoe": "D4",
    "cafe price": "D8",
}

pipelines = {
    # P0: split on spaces, nothing else
    "P0 raw split": lambda t: t.split(),

    # P1: lowercase, letters and digits
    "P1 lowercase": lambda t: re.findall(r"[a-z0-9]+", t.lower()),

    # P2: the full Preprocessor from section 13
    "P2 full pipeline": Preprocessor(),
}

doc_ids = list(documents)
results = {}

for name, analyze in pipelines.items():
    # Index time: every document through this pipeline
    bm25 = BM25Okapi([analyze(documents[d]) for d in doc_ids])

    hits = 0

    for query, expected in queries.items():
        # Query time: the same pipeline
        scores = bm25.get_scores(analyze(query))

        # Top-1 document, or none if nothing scored above 0
        best = doc_ids[scores.argmax()] if scores.max() > 0 else "none"
        hits += best == expected

        print(f"{name:<17} {query!r:<24} top-1 = {best:<5} expected {expected}")

    results[name] = hits
    print()

print({name: f"{h}/{len(queries)}" for name, h in results.items()})

P0 raw split      'PASSWORD e401'          top-1 = none  expected D1
P0 raw split      'resume'                 top-1 = none  expected D2
P0 raw split      'install python 3.12'    top-1 = D3    expected D3
P0 raw split      'run shoe'               top-1 = none  expected D4
P0 raw split      'cafe price'             top-1 = none  expected D8

P1 lowercase      'PASSWORD e401'          top-1 = D1    expected D1
P1 lowercase      'resume'                 top-1 = none  expected D2
P1 lowercase      'install python 3.12'    top-1 = D3    expected D3
P1 lowercase      'run shoe'               top-1 = none  expected D4
P1 lowercase      'cafe price'             top-1 = none  expected D8

P2 full pipeline  'PASSWORD e401'          top-1 = D1    expected D1
P2 full pipeline  'resume'                 top-1 = D2    expected D2
P2 full pipeline  'install python 3.12'    top-1 = D3    expected D3
P2 full pipeline  'run shoe'               top-1 = D4    expected D4
P2 full pipeline  'cafe price'  

```mermaid
%%{init: {'theme':'base','themeVariables':{'primaryColor':'#EEF2FF','primaryBorderColor':'#6366F1','primaryTextColor':'#1E1B4B','lineColor':'#6366F1'}}}%%
xychart-beta
    title "Correct top-1 out of 5 queries"
    x-axis ["P0 raw split", "P1 lowercase", "P2 full pipeline"]
    y-axis "correct" 0 --> 5
    bar [1, 2, 5]
```

| Query | P0 raw split | P1 lowercase | P2 full pipeline | What fixed it |
|---|---|---|---|---|
| `PASSWORD e401` | miss | D1 | D1 | Case folding: `PASSWORD` now meets `Password`. Note `e401` still does not meet `e-401`; section 8 shows how to index both forms |
| `resume` | miss | miss | D2 | Accent folding: without it the P1 regex cuts `résumé` into `r` and `sum` |
| `install python 3.12` | D3 | D3 | D3 | Nothing needed: `python` and `3.12` already match exactly |
| `run shoe` | miss | miss | D4 | Stemming: `running` and `shoes` become `run` and `shoe` |
| `cafe price` | miss | miss | D8 | Accent folding plus stemming `prices` to `price` |

> [!IMPORTANT]
> Same documents, same BM25, same queries: the only change is preprocessing, and correct top-1 answers go from **1 of 5 to 5 of 5**.

## 16. Over-Processing Failures

| Step | Failure | Real example |
|---|---|---|
| **Stemming** | Different words merge | `university` and `universe` to `univers` |
| **Stop words** | Meaning flips | `not prohibited` and `prohibited` become identical |
| **Case folding** | Acronym collisions | `US` and `us`, `IT` and `it` |
| **Digit removal** | Versions and IDs vanish | `Python 3.12` becomes `python` |
| **Punctuation removal** | Codes split | `6ES7513-1AL02-0AB0` becomes three tokens |
| **Accent folding** | Different words merge in some languages | Spanish `año` and `ano` |

```mermaid
%%{init: {'theme':'base','themeVariables':{'primaryColor':'#EEF2FF','primaryBorderColor':'#6366F1','primaryTextColor':'#1E1B4B','lineColor':'#6366F1'}}}%%
flowchart TD
    S["Relevant chunk<br/>not retrieved"] --> A{"Same analyzer<br/>for docs and query?"}
    A -->|No| F1["Use one shared<br/>Preprocessor object"]
    A -->|Yes| B{"Query token<br/>in the index?"}
    B -->|No| C{"Code, version<br/>or number?"}
    C -->|Yes| F2["Protect special<br/>tokens"]
    C -->|No| F3["Check case, accents,<br/>stemming"]
    B -->|Yes| D{"Wrong chunk<br/>ranks higher?"}
    D -->|Yes| F4["Check over-processing:<br/>stop words, stems"]
    D -->|No| F5["Not a preprocessing<br/>problem: check<br/>chunking, embeddings"]
```

## 17. Checkpoints With Answers

### Checkpoint 1 — Fundamentals

**Q1. Why must documents and queries use the same preprocessing?**
Lexical retrieval compares tokens. Different pipelines produce different tokens for the same word, so matches silently fail.

**Q2. What does NFKC do that NFC does not?**
It also maps compatibility characters: ligatures, full-width letters and no-break spaces become their plain forms.

**Q3. Why use `casefold()` instead of `lower()` for matching?**
`casefold()` also handles language-specific cases, for example German `ß` matching `ss`.

### Checkpoint 2 — Tokens

**Q1. Why is a letters-and-digits tokenizer dangerous for RAG?**
It splits versions and codes (`3.12` to `3`, `12`), which are often the most precise part of a query.

**Q2. When are stop words harmful?**
When they carry meaning: negations, phrases like `to be or not to be`, and short queries.

**Q3. Stemming or lemmatization for BM25?**
Usually a light stemmer: fast and good enough. Lemmatization when precision or display quality matters.

### Checkpoint 3 — Sparse vs Dense

**Q1. Should text sent to an embedding model be stemmed?**
No. Embedding models expect natural text; stemming creates words they rarely saw in training.

**Q2. What should be preprocessed for both branches of hybrid search?**
Unicode normalization, whitespace and hyphenation fixes, and protection of codes and versions.

## 18. Interview Questions

#### What is text preprocessing in a RAG system?

All steps that turn raw document and query text into the tokens the retriever stores and compares: Unicode normalization, cleaning, case and accent folding, tokenization, optional stop-word removal and stemming.

#### Why does BM25 need more preprocessing than dense retrieval?

BM25 only matches identical tokens, so every surface difference is a miss. Embedding models map similar text to nearby vectors on their own.

#### What is an analyzer?

The search-engine term for one preprocessing pipeline (character filters, tokenizer, token filters), applied at both index and query time.

#### Stemming vs lemmatization?

Stemming cuts suffixes by rules and can produce non-words or merge unrelated words. Lemmatization maps to dictionary forms using a vocabulary and often the part of speech.

#### How do you handle product codes and versions?

Protect them with a pattern before generic cleaning, index them as whole tokens, and optionally also index their split pieces for partial matches.

#### When would you not remove stop words?

In legal, medical and compliance text (negations), for phrase search, and for any text sent to an embedding model.

## 19. Interview Scenarios

#### Scenario 1 — Search for `résumé` returns nothing

The index folded accents but the query path did not, or the reverse. Fix: one shared analyzer object for both paths.

#### Scenario 2 — Query `Python 3.12` returns 3.11 documents first

The tokenizer split `3.12` into `3` and `12`. Fix: a code-aware tokenizer that keeps versions whole.

#### Scenario 3 — Compliance bot says a product is prohibited when the policy says "not prohibited"

Stop-word removal deleted `not`. Fix: keep negations; never apply stop lists to text sent to the generator or embeddings.

#### Scenario 4 — Search for `universe` returns university pages

The stemmer merged both into `univers`. Fix: a lighter stemmer, a protected-words list, or lemmatization.

## 20. One-Page Revision Map

```mermaid
%%{init: {'theme':'base','themeVariables':{'primaryColor':'#EEF2FF','primaryBorderColor':'#6366F1','primaryTextColor':'#1E1B4B','lineColor':'#6366F1'}}}%%
timeline
    title Text preprocessing at a glance
    Characters : Unicode NFKC : whitespace and hyphenation
    Case and accents : casefold : accent folding
    Tokens : code-aware tokenizer : protect IDs and versions
    Word forms : stop words with care : stemming or lemmatization
    Extras : word n-grams for phrases : char n-grams for typos
    Rule : one analyzer for documents and queries
```

## 21. Practice Tasks

1. Run the section 2 example with your own document and five ways of typing it.
2. Find three ligatures or full-width characters in a real PDF and normalize them with NFKC.
3. Extend `CODE_TOKEN` to keep `#hashtags` and currency amounts such as `$1,299.00`.
4. Write a stop list that keeps all negations, and test it on ten policy sentences.
5. Compare Porter, Snowball and a lemmatizer on 30 words from your own domain.
6. Add a "protected words" list to the `Preprocessor` that is never stemmed.
7. Repeat the section 15 experiment with ten more queries and report correct top-1 per pipeline.
8. Build a typo-tolerant lookup with character trigrams and a Jaccard threshold.
9. Index both the code-aware token and its split pieces, and show which queries improve.
10. Write a unit test that fails if the query path and the index path produce different tokens.

## 22. Final Takeaways

```mermaid
%%{init: {'theme':'base','themeVariables':{'primaryColor':'#EEF2FF','primaryBorderColor':'#6366F1','primaryTextColor':'#1E1B4B','lineColor':'#6366F1'}}}%%
flowchart LR
    subgraph DO["Do"]
        direction TB
        t1["1. Same analyzer for<br/>documents and queries"] --> t2["2. NFKC first"] --> t3["3. casefold and<br/>fold accents for BM25"] --> t4["4. Protect codes,<br/>versions, IDs"] --> t5["5. Light stemming<br/>for BM25"]
    end
    subgraph DONT["Avoid"]
        direction TB
        t6["6. Removing negations"] --> t7["7. Stemming text for<br/>embeddings"] --> t8["8. Letters-only<br/>tokenizers"] --> t9["9. Different pipelines<br/>per side"] --> t10["10. Guessing: measure<br/>top-1 before and after"]
    end
    DO --> DONT
    style DO fill:#F0FDF4,stroke:#16A34A
    style DONT fill:#FEF2F2,stroke:#DC2626
```

> [!TIP]
> **Study rule:** preprocessing is invisible when it works and silent when it fails. Always print the tokens of one document and one query side by side before tuning anything else.